In [20]:
from IPython.display import HTML, display
display(HTML("""
<style>
.reveal .jupyter-widgets.widget-tab { overflow: visible !important; }
.reveal .lm-TabBar, .reveal .p-TabBar {
    display: flex !important;
    min-height: 2.2em !important;
    overflow: visible !important;
}
.reveal .lm-TabBar-content, .reveal .p-TabBar-content { display: flex !important; }
</style>
"""))

In [21]:
import ipywidgets as w
t = w.Tab(children=[w.HTML('A'), w.HTML('B')])
t.titles = ('one', 'two')
t

In [22]:
import ipywidgets as w

tab = w.Tab(children=[
    w.Textarea(value='first panel', description='A:'),
    w.Textarea(value='second panel', description='B:'),
])
tab.titles = ('text one', 'text two')
tab

In [23]:
w.Textarea(value='first panel', description='A:')

Textarea(value='first panel', description='A:')

## Accordion tests
Does an ipywidgets Accordion display in RISE, where Tab loses its tab bar?

In [24]:
# 1. plain ipywidgets Accordion, like the Tab tests above
import ipywidgets as w
acc = w.Accordion(children=[w.HTML('A'), w.HTML('B')], selected_index=0)
acc.titles = ('one', 'two')
acc

Accordion(children=(HTML(value='A'), HTML(value='B')), selected_index=0, titles=('one', 'two'))

In [25]:
# 2. what ModelFlow's rplot shows: modelwidget.tabwidget(tab=False) with svg charts (htmlwidget_fig)
import matplotlib.pyplot as plt
from modelwidget import tabwidget, htmlwidget_fig

figs = {}
for name, k in (('Chart one', 1), ('Chart two', 2)):
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.plot(range(10), [k * x for x in range(10)])
    ax.set_title(name)
    figs[name] = htmlwidget_fig(fig)
    plt.close(fig)

tabwidget(figs, tab=False, selected_index=0).datawidget

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

In [26]:
# 3. nested accordions: how mpak['...'] (visshow) would look with tab=False on both levels
inner = {key: tabwidget({'Charts': figs['Chart one'], 'Data': figs['Chart two']}, tab=False, selected_index=0)
         for key in ('Baseline', 'Alternative', 'Difference')}
tabwidget(inner, tab=False, selected_index=0).datawidget

Accordion(children=(Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!…

## Plain HTML tests (no ipywidgets)
The same switching as Tab and Accordion, but as ordinary HTML output: `<details>` for an accordion and radio buttons + CSS for tabs. No JavaScript, no widget manager.

In [27]:
# helpers: content as html strings, and two ways to switch between them
import uuid
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML
from modelwidget import fig_to_image


def chart(title, k):
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.plot(range(10), [k * x for x in range(10)])
    ax.set_title(title)
    out = fig_to_image(fig, format='svg')
    plt.close(fig)
    return out


def table(k):
    return pd.DataFrame({'year': range(2025, 2030), 'value': [k * x for x in range(5)]}).to_html(index=False)


def html_accordion(parts, open_first=True):
    '''parts: {title: html}; each part a <details> section'''
    name = 'mfa' + uuid.uuid4().hex[:8]   # the same name makes the sections exclusive
    return ''.join(f'<details name="{name}"{" open" if open_first and i == 0 else ""}>'
                   f'<summary style="cursor:pointer;font-weight:bold">{title}</summary>{body}</details>'
                   for i, (title, body) in enumerate(parts.items()))


def html_tabs(parts):
    '''parts: {title: html}; tabs made of hidden radio buttons, labels and CSS'''
    uid = 'mft' + uuid.uuid4().hex[:8]
    css = [f'#{uid} > input {{display:none}}',
           f'#{uid} > label {{display:inline-block;padding:4px 12px;margin-right:2px;cursor:pointer;'
           f'border:1px solid #bbb;border-bottom:none;border-radius:4px 4px 0 0;background:#eee}}',
           f'#{uid} > div {{display:none;border-top:1px solid #bbb;padding-top:6px}}']
    head, panels = '', ''
    for i, (title, body) in enumerate(parts.items()):
        rid = f'{uid}-{i}'
        head += (f'<input type="radio" name="{uid}" id="{rid}"{" checked" if i == 0 else ""}>'
                 f'<label for="{rid}">{title}</label>')
        panels += f'<div>{body}</div>'
        css.append(f'#{rid}:checked + label {{background:#fff;font-weight:bold}}')
        css.append(f'#{rid}:checked ~ div:nth-of-type({i + 1}) {{display:block}}')
    return f'<style>{" ".join(css)}</style><div id="{uid}">{head}{panels}</div>'

print('helpers defined')

helpers defined


In [28]:
# 4. accordion with <details>
HTML(html_accordion({'Chart one': chart('Chart one', 1), 'Chart two': chart('Chart two', 2)}))

In [29]:
# 5. tabs with radio buttons + CSS
HTML(html_tabs({'Chart one': chart('Chart one', 1), 'Chart two': chart('Chart two', 2), 'Data': table(1)}))

year,value
2025,0
2026,1
2027,2
2028,3
2029,4


In [30]:
# 6. nested tabs: how mpak['...'] (visshow) could look without ipywidgets
HTML(html_tabs({key: html_tabs({'Charts': chart(key, k), 'Data': table(k)})
                for k, key in enumerate(('Baseline', 'Alternative', 'Difference'), start=1)}))

year,value
2025,0
2026,1
2027,2
2028,3
2029,4
year,value
2025,0
2026,2
2027,4
2028,6
